In [ ]:
!pip install -U bitsandbytes>=0.46.1

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from tqdm.auto import tqdm
from collections import Counter
import re
import warnings
import os
from sentence_transformers import SentenceTransformer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix
import torch
from transformers import BitsAndBytesConfig, AutoModelForCausalLM, AutoTokenizer
import torch
warnings.filterwarnings('ignore')

In [ ]:
DATASET_PATH = os.environ["DATA_PATH"]
OUTPUT_PATH = os.environ["OUTPUT_PATH"]


data = []
with open(DATASET_PATH, 'r', encoding='utf-8') as f:
    for line in tqdm(f, desc="Загрузка данных"):
        if line.strip():
            data.append(json.loads(line))

print(f"Загружено {len(data)} пар предложений")

df = pd.DataFrame(data)
print(f"\nРазмер датафрейма: {df.shape}")
print(f"\nКолонки: {df.columns.tolist()}")

In [ ]:
all_data = []
with open(DATASET_PATH, 'r', encoding='utf-8') as f:
    for line in f:
        if line.strip():
            try:
                all_data.append(json.loads(line))
            except:
                continue

In [ ]:
MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token

In [ ]:
def generate(PROMPT, max_tokens=1024, retries=3):
    for attempt in range(retries):
        try:
            messages = [
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": PROMPT}
            ]

            text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
            inputs = tokenizer([text], return_tensors="pt").to(model.device)

            with torch.no_grad():
                outputs = model.generate(
                    **inputs,
                    max_new_tokens=max_tokens,
                    temperature=0.1,
                    do_sample=True,
                    pad_token_id=tokenizer.pad_token_id,
                    eos_token_id=tokenizer.eos_token_id,
                )

            response = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)

            match = re.search(r'\{.*\}', response, re.DOTALL)
            if not match:
                if attempt < retries - 1:
                    continue
                return None

            try:
                return json.loads(match.group())
            except json.JSONDecodeError:
                if attempt < retries - 1:
                    continue
                fixed = match.group().replace('\n', '\\n').replace('\r', '\\r')
                try:
                    return json.loads(fixed)
                except:
                    return None

        except Exception:
            if attempt < retries - 1:
                continue
            return None

    return None

In [ ]:
def preprocess_text(text):
    if not text or not isinstance(text, str):
        return ""

    text = text.strip()

    text = re.sub(r'\s+', ' ', text)

    text = re.sub(r'[ \t]+', ' ', text)

    text = re.sub(r'\n{3,}', '\n\n', text)

    text = re.sub(r'[^\x00-\x7F\x80-\xFF\u0400-\u04FF]', '', text)

    return text.strip()


def preprocess_pair(item):
    processed = item.copy()

    if 'en' in processed:
        processed['en'] = preprocess_text(processed['en'])
    if 'ru' in processed:
        processed['ru'] = preprocess_text(processed['ru'])

    return processed

In [ ]:
SYSTEM_PROMPT = f"""
You are a data cleaner. Find and remove EXACT garbage patterns.

CRITICAL RULES:
- You can ONLY REMOVE or FIX garbage
- You CANNOT rephrase or improve style
- You CANNOT add words that weren't there
- If no garbage found, return EXACT original

GARBAGE PATTERNS:

1. SLIPPED WORDS / EXTRA LETTERS (CRITICAL):
   - "Companyss" → "Company" (extra 's')
   - "JohnBusiness" → "John Business" (missing space)
   - "interestbearing" → "interest-bearing" (missing hyphen)

2. REPEATED WORDS:
   - "Provisions Provisions" → "Provisions"

3. REPEATED PHRASES:
   - Any 3+ word sequence repeated twice

4. OCR ERRORS:
   - "Deve1opment" → "Development"
   - "stresss" → "stresses"
   - Numbers inside words

5. RANDOM TEXT:
   - "Eldonakeg ueat." → remove (nonsense)
   - Keyboard mash: "asdfghjkl", "qwerty"

6. GERMAN WORDS:
   - der, die, das, und, für, mit, von, zu

7. CUT-OFF TEXT:
   - Random fragments at start: "указанным выше. Мы" → "Мы"

8. EXTRA CHARACTERS:
   - "* " at start → remove

HOW TO DESCRIBE CHANGES:
- For fixing a word: "old_word → new_word"
- For removing text: "removed 'text'"
- For adding punctuation: "added '-' to 'interestbearing' → 'interest-bearing'"
- BE SPECIFIC and CLEAR

EXAMPLES OF GOOD CHANGE DESCRIPTIONS:
- "Companyss management → Company management"
- "interestbearing → interest-bearing"
- "removed 'указанным выше.' from start"
- "removed repeated 'Provisions'"

DO NOT CHANGE:
- Word order, style, prepositions, correct grammar, terminology
- DO NOT invent changes

Return JSON:
{{
    "clean_en": "exact text with garbage removed",
    "clean_ru": "exact text with garbage removed",
    "was_cleaned": true/false,
    "changes": ["clear description of each change"]
}}
"""

In [ ]:
def clean_pair(en_text, ru_text, domain=""):
    en_text = preprocess_text(en_text)
    ru_text = preprocess_text(ru_text)

    if not en_text or not ru_text:
        return {
            'en': en_text,
            'ru': ru_text,
            'was_cleaned': False,
            'is_aligned': False,
            'alignment_issue': 'Empty text',
            'is_fixable': False,
            'unfixable_reason': 'Empty text',
            'changes': [],
            'error': True
        }

    PROMPT = f"""
Process this pair:
EN: {en_text}
RU: {ru_text}
"""

    result = generate(PROMPT, max_tokens=1024, retries=3)

    if not result:
        return {
            'en': en_text,
            'ru': ru_text,
            'was_cleaned': False,
            'is_aligned': True,
            'alignment_issue': '',
            'is_fixable': True,
            'unfixable_reason': '',
            'changes': [],
            'error': True
        }

    clean_en = result.get('clean_en', en_text)
    clean_ru = result.get('clean_ru', ru_text)
    was_cleaned = result.get('was_cleaned', False)
    is_aligned = result.get('is_aligned', True)
    alignment_issue = result.get('alignment_issue', '')
    is_fixable = result.get('is_fixable', True)
    unfixable_reason = result.get('unfixable_reason', '')
    changes = result.get('changes', [])

    if changes:
        filtered_changes = []
        for change in changes:
            if not change or not change.strip():
                continue
            if any(word in change.lower() for word in ['kept as is', 'no change', 'unchanged']):
                continue
            if 'replaced' in change.lower() and 'with' in change.lower():
                parts = change.split('with')
                if len(parts) == 2:
                    before = parts[0].replace('Replaced', '').replace('replaced', '').strip(" '\"")
                    after = parts[1].strip(" '\"")
                    if before == after:
                        continue
            filtered_changes.append(change)
        changes = filtered_changes

    if not changes:
        was_cleaned = False
        clean_en = en_text
        clean_ru = ru_text

    return {
        'en': clean_en,
        'ru': clean_ru,
        'was_cleaned': was_cleaned,
        'is_aligned': is_aligned,
        'alignment_issue': alignment_issue,
        'is_fixable': is_fixable,
        'unfixable_reason': unfixable_reason,
        'changes': changes,
        'error': False
    }

In [ ]:
def process_pairs(data_list, checkpoint_file='checkpoint.json'):
    results = []

    try:
        with open(checkpoint_file, 'r', encoding='utf-8') as f:
            results = json.load(f)
        start = len(results)
        print(f"Loaded {start} results from checkpoint")
    except:
        results = []
        start = 0
        print("Starting from scratch")

    remaining_data = data_list[start:]

    if not remaining_data:
        print("All pairs already processed")
        return results

    iterator = tqdm(remaining_data, desc="Processing pairs", unit="pair", initial=start, total=len(data_list))

    for idx, item in enumerate(iterator):
        en_text = item.get('en', '')
        ru_text = item.get('ru', '')
        domain = item.get('domain', '')
        doc_name = item.get('doc_name', '')

        item = preprocess_pair(item)

        result = clean_pair(item.get('en', ''), item.get('ru', ''), domain)
        result['domain'] = domain
        result['doc_name'] = doc_name

        results.append(result)

        if (start + idx + 1) % 10 == 0:
            with open(checkpoint_file, 'w', encoding='utf-8') as f:
                json.dump(results, f, ensure_ascii=False, indent=2)

        total = len(results)
        cleaned = sum(1 for r in results if r.get('was_cleaned', False))
        aligned = sum(1 for r in results if r.get('is_aligned', True))
        fixable = sum(1 for r in results if r.get('is_fixable', True))
        errors = sum(1 for r in results if r.get('error', False))

        iterator.set_postfix_str(f"clean:{cleaned} aligned:{aligned} fixable:{fixable} errors:{errors}")

    with open(checkpoint_file, 'w', encoding='utf-8') as f:
        json.dump(results, f, ensure_ascii=False, indent=2)

    return results

In [ ]:
output_root = Path(f"../{OUTPUT_PATH}").resolve()


if OUTPUT_PATH and os.path.exists(OUTPUT_PATH):
    os.remove(OUTPUT_PATH)
    print("Checkpoint deleted")

for f in ['clean_dataset.json', 'bad_pairs.json', 'full_results.json']:
    path = output_root / f
    if path.exists():
        path.unlink()
        print(f"Deleted {f}")

results = process_pairs(all_data, checkpoint_file=OUTPUT_PATH)